In [34]:
import pandas as pd
import requests
import json
import os
import uuid
import hashlib
from datetime import datetime, date, timezone
from google.cloud import bigquery

In [17]:
exchange_rates_url = "https://api.exchangeratesapi.io/latest"  
exchange_rates_api_key = 'e09d4c9bbb3a9d7c1e04ec7b8c91052a'  #os.getenv("EXCHANGE_RATES_API_KEY")
ecb_url = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml"


In [29]:
r = requests.get(url=exchange_rates_url, params={"access_key": exchange_rates_api_key})
#data = 
print(r.json())
raw_json = json.dumps(r.json(), sort_keys=True)
raw_json
hashlib.sha256(raw_json.encode("utf-8")).hexdigest()

#print(data)
#df = pd.DataFrame(data['rates'], index=[0])
#rint(df)

{'success': True, 'timestamp': 1757949545, 'base': 'EUR', 'date': '2025-09-15', 'rates': {'AED': 4.319418, 'AFN': 80.101141, 'ALL': 96.887667, 'AMD': 450.607541, 'ANG': 2.105786, 'AOA': 1078.531386, 'ARS': 1725.752315, 'AUD': 1.765067, 'AWG': 2.117074, 'AZN': 2.001176, 'BAM': 1.956561, 'BBD': 2.368021, 'BDT': 143.114421, 'BGN': 1.956718, 'BHD': 0.443489, 'BIF': 3508.964162, 'BMD': 1.176152, 'BND': 1.506786, 'BOB': 8.142165, 'BRL': 6.253488, 'BSD': 1.175757, 'BTC': 1.0272476e-05, 'BTN': 103.622285, 'BWP': 16.609457, 'BYN': 3.981148, 'BYR': 23052.584037, 'BZD': 2.364619, 'CAD': 1.623778, 'CDF': 3334.39163, 'CHF': 0.93469, 'CLF': 0.028519, 'CLP': 1118.767632, 'CNY': 8.379737, 'CNH': 8.373022, 'COP': 4580.383787, 'CRC': 592.230238, 'CUC': 1.176152, 'CUP': 31.168035, 'CVE': 110.306946, 'CZK': 24.328416, 'DJF': 209.371396, 'DKK': 7.464609, 'DOP': 74.428935, 'DZD': 152.466937, 'EGP': 56.656072, 'ERN': 17.642284, 'ETB': 169.444774, 'EUR': 1, 'FJD': 2.63011, 'FKP': 0.86798, 'GBP': 0.865319, 'GE

'0c4718567384502012199148663656164218f0225417fbdce9d76897041527b9'

In [35]:
def parse_exchangeratesapi():
    r = requests.get(url=exchange_rates_url, params={"access_key": exchange_rates_api_key})
    payload = r.json()
    raw_json = json.dumps(payload, sort_keys=True)
    raw_hash = hashlib.sha256(raw_json.encode("utf-8")).hexdigest()
    rates = payload.get("rates", {})
    rate_date = payload.get("date")
    rows = []
    for quote, r in rates.items():
        rows.append({
            "id": str(uuid.uuid4()),
            "source": "exchangeratesapi",
            "source_id": payload.get("timestamp") and str(payload.get("timestamp")),
            "base_currency": 'EUR',
            "quote_currency": quote,
            "rate": float(r),
            "rate_date": pd.to_datetime(rate_date).date(),
            "rate_timestamp": None,
            "retrieved_at": datetime.utcnow().isoformat(),
            "raw_hash": raw_hash,
            "raw_payload": raw_json,
            "ingest_job_id": None,  # set later
            "created_at": datetime.utcnow().isoformat()
        })
    return pd.DataFrame(rows)

df = parse_exchangeratesapi()
print(df)

                                       id            source   source_id  \
0    19368117-1446-4ee5-bcb7-557adeb0f8c5  exchangeratesapi  1757949545   
1    82391b48-8670-443f-a0cb-4fdfce6fb0dd  exchangeratesapi  1757949545   
2    bdf3ebb7-1567-4cb2-9fc5-84c93c95f965  exchangeratesapi  1757949545   
3    4e057a31-86b3-47bf-923b-160a45c40db9  exchangeratesapi  1757949545   
4    c8242fdc-d821-49e4-9133-d555181b0391  exchangeratesapi  1757949545   
..                                    ...               ...         ...   
167  cb92d2a1-e0aa-4e18-9324-4d513110985d  exchangeratesapi  1757949545   
168  d5d47175-b1e3-4928-82a0-395356ea2a05  exchangeratesapi  1757949545   
169  92502206-bd93-47ac-b061-4935036d2986  exchangeratesapi  1757949545   
170  e25d8215-077d-4c5e-8850-3171224d4e4a  exchangeratesapi  1757949545   
171  06a11e93-e95b-4a2f-986c-29031b31383f  exchangeratesapi  1757949545   

    base_currency quote_currency          rate   rate_date rate_timestamp  \
0             EUR     

In [21]:
r = requests.get(ecb_url, timeout=30)
ecb_data = r.text
print(ecb_data)
#df_ecb = pd.DataFrame(ecb_data['rates'], index=[0])
#print(df)

<?xml version="1.0" encoding="UTF-8"?>
<gesmes:Envelope xmlns:gesmes="http://www.gesmes.org/xml/2002-08-01" xmlns="http://www.ecb.int/vocabulary/2002-08-01/eurofxref">
	<gesmes:subject>Reference rates</gesmes:subject>
	<gesmes:Sender>
		<gesmes:name>European Central Bank</gesmes:name>
	</gesmes:Sender>
	<Cube>
		<Cube time='2025-09-15'>
			<Cube currency='USD' rate='1.1766'/>
			<Cube currency='JPY' rate='173.38'/>
			<Cube currency='BGN' rate='1.9558'/>
			<Cube currency='CZK' rate='24.327'/>
			<Cube currency='DKK' rate='7.4646'/>
			<Cube currency='GBP' rate='0.86410'/>
			<Cube currency='HUF' rate='389.93'/>
			<Cube currency='PLN' rate='4.2505'/>
			<Cube currency='RON' rate='5.0620'/>
			<Cube currency='SEK' rate='10.9115'/>
			<Cube currency='CHF' rate='0.9353'/>
			<Cube currency='ISK' rate='143.20'/>
			<Cube currency='NOK' rate='11.5565'/>
			<Cube currency='TRY' rate='48.5585'/>
			<Cube currency='AUD' rate='1.7659'/>
			<Cube currency='BRL' rate='6.2769'/>
			<Cube currency